# Reading data from Adls



In [0]:
df = spark.read.format("parquet").load(
    "abfss://bronze-layer@adlcarproject.dfs.core.windows.net/Raw_data"
)
df.display()

#Importing Required Module

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import  *


%md
#Transformation stage 


In [0]:
df1 = df.withColumn("Model_Category",split(col("Model_id"),"-")[0])

In [0]:
df1.display()

In [0]:
df2 = df1.withColumn("Units_Sold", col("Units_Sold").cast(LongType()))
df2.printSchema()

In [0]:
df3 = df2.withColumn("Single_unit_cost",col("Revenue")/col("Units_Sold"))

In [0]:
df3 = df3.sort("Units_sold", ascending=False)
df3.display()


In [0]:
from pyspark.sql.functions import sum

df4 = df3.groupBy(["Year", "BranchName"]).agg(
    sum("Units_Sold").alias("Total_Units")
).sort(["Year","Total_Units"],ascending = [1,0])



Databricks visualization. Run in Databricks to view.

# Writing Data into the Silver layer


In [0]:
df3.write.format("parquet") \
    .mode("overwrite") \
    .save("abfss://silver-layer@adlcarproject.dfs.core.windows.net/carsalesdata")

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver-layer@adlcarproject.dfs.core.windows.net/carsalesdata`